To do:

- Filter for fbs teams only !
- Get rid of test/ train aspect (train on 2020-2024 and predict on given 2025 game id entry) !
- validate the 2025 game id entry for the text box
- get visualizations and predictions as output on the webpage
- make webpage look nice

In [1]:
# import library
import streamlit as st
import os
import tabulate
import requests
import pandas as pd
import numpy as np
import pyarrow
import sportsdataverse as sdv
import polars as pl
from great_tables import GT, md
from scipy import stats
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss

---
Win probability model Code

In [2]:
# Set seed for reproducibility
np.random.seed(36)

In [3]:
CFBD_API_KEY = "cKf6FOVq1WGS2xOlvMoAMw6TB3ELUG7vX1h6FoQcqBceFCHHfJBDfs+WrpABR8xy"
headers = {"Authorization": f"Bearer {CFBD_API_KEY}"}

In [4]:
seasons = range(2020, 2023) 
weeks = range(1, 16)
raw_stats_list = []

# Data ingestion loop
for yr in seasons:
    for wk in weeks:
        url = f"https://api.collegefootballdata.com/games/teams?year={yr}&week={wk}&seasonType=regular&division=fbs"
        try:
            response = requests.get(url, headers=headers, timeout=10)
            if response.status_code == 200:
                data = response.json()
                for game in data:
                    game_id = game.get("id")
                    teams = game.get("teams", [])
                    if len(teams) == 2:
                        for idx, team_data in enumerate(teams):
                            opp_data = teams[1 - idx]
                            
                            # Helper to extract metric by name
                            def get_stat(t_data, stat_name):
                                for s in t_data.get("stats", []):
                                    if s.get("category") == stat_name:
                                        return s.get("stat")
                                return None

                            row = {
                                "game_id": game_id,
                                "season": str(yr),
                                "week_num": wk,
                                "team": team_data.get("team"),
                                "opponent": opp_data.get("team"),
                                "home_away": "home" if team_data.get("homeAway") == "home" else "away",
                                "points": team_data.get("points"),
                                "points_allowed": opp_data.get("points"),
                                "total_yards": get_stat(team_data, "totalYards"),
                                "total_yards_allowed": get_stat(opp_data, "totalYards"),
                                "net_passing_yards": get_stat(team_data, "netPassingYards"),
                                "net_passing_yards_allowed": get_stat(opp_data, "netPassingYards"),
                                "rushing_yards": get_stat(team_data, "rushingYards"),
                                "rushing_yards_allowed": get_stat(opp_data, "rushingYards"),
                                "turnovers": get_stat(team_data, "turnovers"),
                                "turnovers_allowed": get_stat(opp_data, "turnovers"),
                                "first_downs": get_stat(team_data, "firstDowns"),
                                "first_downs_allowed": get_stat(opp_data, "firstDowns"),
                                "third_down_eff": get_stat(team_data, "thirdDownEff"),
                                "third_down_eff_allowed": get_stat(opp_data, "thirdDownEff"),
                                "total_penalties_yards": get_stat(team_data, "totalPenaltiesYards"),
                                "total_penalties_yards_allowed": get_stat(opp_data, "totalPenaltiesYards"),
                                "possession_time": get_stat(team_data, "possessionTime"),
                                "possession_time_allowed": get_stat(opp_data, "possessionTime"),
                            }
                            raw_stats_list.append(row)
        except Exception as e:
            continue

raw_team_stats = pd.DataFrame(raw_stats_list)

In [5]:
raw_team_stats['season'].value_counts()

season
2022    3064
2021    1698
2020    1046
Name: count, dtype: int64

In [6]:
seasons = range(2023, 2026) 
weeks = range(1, 16)
raw_stats_list2 = []

# Data ingestion loop
for yr in seasons:
    for wk in weeks:
        url = f"https://api.collegefootballdata.com/games/teams?year={yr}&week={wk}&seasonType=regular&division=fbs"
        try:
            response = requests.get(url, headers=headers, timeout=10)
            if response.status_code == 200:
                data = response.json()
                for game in data:
                    game_id = game.get("id")
                    teams = game.get("teams", [])
                    if len(teams) == 2:
                        for idx, team_data in enumerate(teams):
                            opp_data = teams[1 - idx]
                            
                            # Helper to extract metric by name
                            def get_stat(t_data, stat_name):
                                for s in t_data.get("stats", []):
                                    if s.get("category") == stat_name:
                                        return s.get("stat")
                                return None

                            row = {
                                "game_id": game_id,
                                "season": str(yr),
                                "week_num": wk,
                                "team": team_data.get("team"),
                                "opponent": opp_data.get("team"),
                                "home_away": "home" if team_data.get("homeAway") == "home" else "away",
                                "points": team_data.get("points"),
                                "points_allowed": opp_data.get("points"),
                                "total_yards": get_stat(team_data, "totalYards"),
                                "total_yards_allowed": get_stat(opp_data, "totalYards"),
                                "net_passing_yards": get_stat(team_data, "netPassingYards"),
                                "net_passing_yards_allowed": get_stat(opp_data, "netPassingYards"),
                                "rushing_yards": get_stat(team_data, "rushingYards"),
                                "rushing_yards_allowed": get_stat(opp_data, "rushingYards"),
                                "turnovers": get_stat(team_data, "turnovers"),
                                "turnovers_allowed": get_stat(opp_data, "turnovers"),
                                "first_downs": get_stat(team_data, "firstDowns"),
                                "first_downs_allowed": get_stat(opp_data, "firstDowns"),
                                "third_down_eff": get_stat(team_data, "thirdDownEff"),
                                "third_down_eff_allowed": get_stat(opp_data, "thirdDownEff"),
                                "total_penalties_yards": get_stat(team_data, "totalPenaltiesYards"),
                                "total_penalties_yards_allowed": get_stat(opp_data, "totalPenaltiesYards"),
                                "possession_time": get_stat(team_data, "possessionTime"),
                                "possession_time_allowed": get_stat(opp_data, "possessionTime"),
                            }
                            raw_stats_list2.append(row)
        except Exception as e:
            continue

raw_team_stats2 = pd.DataFrame(raw_stats_list2)

In [7]:
raw_team_stats2['season'].value_counts()

season
2025    3250
2024    3210
2023    2980
Name: count, dtype: int64

In [8]:
raw_team_stats = pd.concat([raw_team_stats, raw_team_stats2], ignore_index=True)

In [9]:
raw_team_stats['season'].value_counts()

season
2025    3250
2024    3210
2022    3064
2023    2980
2021    1698
2020    1046
Name: count, dtype: int64

In [10]:
# Filter for FBS teams only
# Create a list of all FBS schools
fbs_teams = ["Boston College", "California", "Clemson", "Duke", "Florida State",
    "Georgia Tech", "Louisville", "Miami", "NC State", "North Carolina",
    "Pittsburgh", "SMU", "Stanford", "Syracuse", "Virginia",
    "Virginia Tech", "Wake Forest", "Illinois", "Indiana", "Iowa", "Maryland", "Michigan",
    "Michigan State", "Minnesota", "Nebraska", "Northwestern", "Ohio State",
    "Oregon", "Penn State", "Purdue", "Rutgers", "UCLA",
    "USC", "Washington", "Wisconsin", "Arizona", "Arizona State", "Baylor", "BYU", "Cincinnati",
    "Colorado", "Houston", "Iowa State", "Kansas", "Kansas State",
    "Oklahoma State", "TCU", "Texas Tech", "UCF", "Utah",
    "West Virginia", "Alabama", "Arkansas", "Auburn", "Florida", "Georgia",
    "Kentucky", "LSU", "Mississippi State", "Missouri", "Oklahoma",
    "Ole Miss", "South Carolina", "Tennessee", "Texas", "Texas A&M",
    "Vanderbilt", "Army", "Charlotte", "East Carolina", "Florida Atlantic", "Memphis",
    "Navy", "North Texas", "Rice", "Temple", "Tulane",
    "Tulsa", "UAB", "South Florida", "UTSA", "Boise State", "Colorado State", "Fresno State",
    "Oregon State", "San Diego State", "Texas State", "Utah State", "Washington State",
    "Air Force", "Hawai'i", "Nevada", "New Mexico", "North Dakota State",
    "Northern Illinois", "San José State", "UNLV", "UTEP", "Wyoming",
    "Akron", "Ball State", "Bowling Green", "Buffalo", "Central Michigan",
    "Eastern Michigan", "Kent State", "Miami (OH)", "Ohio", "Sacramento State",
    "Toledo", "Massachusetts", "Western Michigan",
    "Delaware", "Florida International", "Jacksonville State", "Kennesaw State", "Liberty",
    "Middle Tennessee", "Missouri State", "New Mexico State", "Sam Houston", "Western Kentucky",
    "App State", "Arkansas State", "Coastal Carolina", "Georgia Southern", "Georgia State",
    "James Madison", "Louisiana", "Louisiana Tech", "Marshall", "Old Dominion",
    "South Alabama", "Southern Miss", "Troy", "UL Monroe", "Notre Dame", "UConn"
]

# Now, let's filter the data so that it only keeps fbs vs fbs games
# and it only keeps the regular season games and plays that we care about on offense.
raw_team_stats = raw_team_stats[(raw_team_stats['team'].isin(fbs_teams)) & (raw_team_stats['opponent'].isin(fbs_teams))
].copy()

In [11]:
def parse_time_sec(time_str):
    if pd.isna(time_str):
        return 1800.0
    parts = str(time_str).split(":")
    if len(parts) == 2:
        return float(parts[0]) * 60 + float(parts[1])
    return 1800.0

def parse_split_eff(eff_str, pos=0):
    if pd.isna(eff_str):
        return 0.0
    parts = str(eff_str).split("-")
    if len(parts) == 2:
        try:
            return float(parts[pos])
        except ValueError:
            return 0.0
    return 0.0



In [12]:
df = raw_team_stats.copy()
df = df.dropna(subset=["points", "points_allowed"])
df = df[df["points"] != df["points_allowed"]]

df["points"] = df["points"].astype(float)
df["points_allowed"] = df["points_allowed"].astype(float)
df["win"] = (df["points"] > df["points_allowed"]).astype(int)

# Parsing nested string fields
df["td_comp"] = df["third_down_eff"].apply(lambda x: parse_split_eff(x, 0))
df["td_att"] = df["third_down_eff"].apply(lambda x: parse_split_eff(x, 1))
df["td_comp_opp"] = df["third_down_eff_allowed"].apply(lambda x: parse_split_eff(x, 0))
df["td_att_opp"] = df["third_down_eff_allowed"].apply(lambda x: parse_split_eff(x, 1))

df["pen_yds"] = df["total_penalties_yards"].apply(lambda x: parse_split_eff(x, 1))
df["pen_yds_opp"] = df["total_penalties_yards_allowed"].apply(lambda x: parse_split_eff(x, 1))

df["top_sec"] = df["possession_time"].apply(parse_time_sec)
df["top_sec_opp"] = df["possession_time_allowed"].apply(parse_time_sec)

df["third_down_pct"] = np.where(df["td_att"] > 0, df["td_comp"] / df["td_att"], 0.0)
df["third_down_pct_opp"] = np.where(df["td_att_opp"] > 0, df["td_comp_opp"] / df["td_att_opp"], 0.0)

num_cols = ["turnovers", "turnovers_allowed", "net_passing_yards", "net_passing_yards_allowed",
            "rushing_yards", "rushing_yards_allowed", "first_downs", "first_downs_allowed"]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

In [13]:
# Computing differentials
df["turnover_margin"] = df["turnovers_allowed"] - df["turnovers"]
df["pass_yards_diff"] = df["net_passing_yards"] - df["net_passing_yards_allowed"]
df["rush_yards_diff"] = df["rushing_yards"] - df["rushing_yards_allowed"]
df["first_downs_diff"] = df["first_downs"] - df["first_downs_allowed"]
df["third_down_pct_diff"] = df["third_down_pct"] - df["third_down_pct_opp"]
df["penalty_yards_diff"] = df["pen_yds"] - df["pen_yds_opp"]
df["top_seconds_diff"] = df["top_sec"] - df["top_sec_opp"]
df["is_home"] = (df["home_away"].str.lower() == "home").astype(int)

# Filter every 2nd row to prevent double-counting full game perspectives
cfb_diff = df.iloc[1::2].reset_index(drop=True)

diff_predictors = [
    "is_home", "turnover_margin", "pass_yards_diff", "rush_yards_diff",
    "first_downs_diff", "third_down_pct_diff", "penalty_yards_diff", "top_seconds_diff"
]

cfb_diff = cfb_diff.dropna(subset=["win"] + diff_predictors)
cfb_diff.head()

,game_id,season,week_num,team,opponent,home_away,points,points_allowed,total_yards,total_yards_allowed,...,third_down_pct,third_down_pct_opp,turnover_margin,pass_yards_diff,rush_yards_diff,first_downs_diff,third_down_pct_diff,penalty_yards_diff,top_seconds_diff,is_home
0,401234576,2020,1,BYU,Navy,away,55.0,3.0,580,149,...,0.545455,0.166667,0.0,249.0,182.0,21.0,0.378788,-49.0,880.0,0
1,401235700,2020,1,Middle Tennessee,Army,away,0.0,42.0,184,368,...,0.333333,0.866667,-4.0,81.0,-265.0,-11.0,-0.533333,-21.0,-658.0,0
2,401207101,2020,1,Southern Miss,South Alabama,home,21.0,32.0,409,526,...,0.357143,0.583333,2.0,-49.0,-68.0,-3.0,-0.226190,11.0,104.0,1
3,401212553,2020,1,Arkansas State,Memphis,away,24.0,37.0,424,498,...,0.400000,0.529412,-2.0,24.0,-98.0,-4.0,-0.129412,-52.0,-501.0,0
4,401212484,2020,1,Texas State,SMU,home,24.0,31.0,416,544,...,0.375000,0.428571,1.0,-140.0,12.0,-4.0,-0.053571,-20.0,-206.0,1


### Split the data into the training (2020-2024) and the prediction (2025) set(s).

In [14]:
cfb_diff['season'] = cfb_diff['season'].astype(int)

In [15]:
cfb_diff['season'].value_counts()

season
2025    762
2024    756
2023    755
2022    742
2021    738
2020    492
Name: count, dtype: int64

In [16]:
X_diff = cfb_diff[diff_predictors]
y_diff = cfb_diff["win"]
strata_diff = cfb_diff["season"]

'''X_train, X_test, y_train, y_test = train_test_split(
    X_diff, y_diff, test_size=0.20, random_state=2026, stratify=strata_diff
)'''

train_data = cfb_diff[cfb_diff['season'].isin([2020, 2021, 2022, 2023, 2024])]
test_data = cfb_diff[cfb_diff['season'] == 2025]
X_train = train_data[diff_predictors]
y_train = train_data["win"]
X_test = test_data[diff_predictors]
y_test = test_data["win"]

'''test_indices = X_test.index
test_seasons = cfb_diff.loc[test_indices, "season"]'''

# Standardizing features
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=diff_predictors, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=diff_predictors, index=X_test.index)

# Add constant for statsmodels Logit
X_train_sm = sm.add_constant(X_train_scaled)
X_test_sm = sm.add_constant(X_test_scaled)


In [17]:
# Fit Logistic Regression Model
logit_model_diff = sm.Logit(y_train, X_train_sm).fit(disp=False)

# Feature Importance Summary Table
summary_df = pd.DataFrame({
    "Predictor": logit_model_diff.params.index,
    "Std Beta": logit_model_diff.params.values,
    "Odds Ratio (1 SD)": np.exp(logit_model_diff.params.values),
    "z-statistic": logit_model_diff.tvalues.values,
    "p-value": logit_model_diff.pvalues.values
})
summary_df = summary_df[summary_df["Predictor"] != "const"].copy()
summary_df["Importance (|z|)"] = summary_df["z-statistic"].abs()
summary_df = summary_df.sort_values(by="Importance (|z|)", ascending=False)

print(summary_df.to_string(index=False))

          Predictor  Std Beta  Odds Ratio (1 SD)  z-statistic       p-value  Importance (|z|)
    turnover_margin  1.789331           5.985446    21.865137 5.579816e-106         21.865137
    rush_yards_diff  2.549930          12.806213    19.531835  5.888459e-85         19.531835
    pass_yards_diff  1.656232           5.239531    14.372402  7.710947e-47         14.372402
third_down_pct_diff  0.961287           2.615060    11.761033  6.197122e-32         11.761033
 penalty_yards_diff -0.422508           0.655401    -6.610948  3.818668e-11          6.610948
            is_home  0.248790           1.282473     4.432780  9.302590e-06          4.432780
   top_seconds_diff -0.355549           0.700789    -4.282523  1.847858e-05          4.282523
   first_downs_diff -0.542341           0.581386    -4.015003  5.944494e-05          4.015003


Calculate the Win probability (need to change get rid of the test/train aspect).

In [18]:
pred_probs_diff = logit_model_diff.predict(X_test_sm)
pred_class_diff = (pred_probs_diff >= 0.5).astype(int)

metrics_diff = pd.DataFrame({
    "Metric": ["Accuracy", "ROC AUC", "Log Loss", "Brier Score"],
    "Test Score": [
        accuracy_score(y_test, pred_class_diff),
        roc_auc_score(y_test, pred_probs_diff),
        log_loss(y_test, pred_probs_diff),
        brier_score_loss(y_test, pred_probs_diff)
    ]
})

print("Out-of-Sample Performance (Differentials Model):")
print(metrics_diff.to_string(index=False))

Out-of-Sample Performance (Differentials Model):
     Metric  Test Score
   Accuracy    0.864829
    ROC AUC    0.944105
   Log Loss    0.301421
Brier Score    0.095659


In [19]:
cfb_diff_2025 = cfb_diff[cfb_diff['season'] == 2025].copy()

In [20]:
cfb_diff_2025

,game_id,season,week_num,team,opponent,home_away,points,points_allowed,total_yards,total_yards_allowed,...,third_down_pct,third_down_pct_opp,turnover_margin,pass_yards_diff,rush_yards_diff,first_downs_diff,third_down_pct_diff,penalty_yards_diff,top_seconds_diff,is_home
3483,401756846,2025,1,Kansas State,Iowa State,home,21.0,24.0,383,313,...,0.384615,0.214286,0.0,90.0,-20.0,-6.0,0.170330,43.0,-464.0,1
3484,401752667,2025,1,Baylor,Auburn,home,24.0,38.0,483,415,...,0.333333,0.500000,0.0,311.0,-243.0,-2.0,-0.166667,-28.0,-396.0,1
3485,401756847,2025,1,Kansas,Fresno State,home,31.0,7.0,383,216,...,0.200000,0.466667,3.0,-3.0,170.0,9.0,-0.266667,22.0,376.0,1
3486,401754523,2025,1,TCU,North Carolina,away,48.0,14.0,542,222,...,0.583333,0.100000,2.0,112.0,208.0,19.0,0.483333,33.0,258.0,0
3487,401757218,2025,1,Western Kentucky,Sam Houston,home,41.0,24.0,506,382,...,0.578947,0.142857,-1.0,192.0,-68.0,12.0,0.436090,-34.0,352.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4240,401777353,2025,15,Indiana,Ohio State,away,13.0,10.0,340,322,...,0.461538,0.333333,0.0,-42.0,60.0,0.0,0.128205,29.0,-26.0,0
4241,401777329,2025,15,Miami (OH),Western Michigan,away,13.0,23.0,272,397,...,0.400000,0.266667,-1.0,88.0,-213.0,0.0,0.133333,13.0,-634.0,0
4242,401777351,2025,15,Georgia,Alabama,away,28.0,7.0,297,209,...,0.375000,0.214286,1.0,-56.0,144.0,5.0,0.160714,75.0,824.0,0
4243,401777328,2025,15,Duke,Virginia,away,27.0,20.0,333,344,...,0.375000,0.466667,1.0,-20.0,9.0,-7.0,-0.091667,32.0,528.0,0


In [21]:
pred_probs_diff

3483    0.888079
3484    0.441562
3485    0.985852
3486    0.999592
3487    0.883266
          ...   
4240    0.655756
4241    0.035404
4242    0.858471
4243    0.498065
4244    0.980450
Length: 762, dtype: float64

In [22]:
# Merge predicitons with the 2025 data for display
predictions = pd.merge(cfb_diff_2025, pd.DataFrame({'predicted_win_prob': pred_probs_diff}), left_index=True, right_index=True)

In [23]:
predictions

,game_id,season,week_num,team,opponent,home_away,points,points_allowed,total_yards,total_yards_allowed,...,third_down_pct_opp,turnover_margin,pass_yards_diff,rush_yards_diff,first_downs_diff,third_down_pct_diff,penalty_yards_diff,top_seconds_diff,is_home,predicted_win_prob
3483,401756846,2025,1,Kansas State,Iowa State,home,21.0,24.0,383,313,...,0.214286,0.0,90.0,-20.0,-6.0,0.170330,43.0,-464.0,1,0.888079
3484,401752667,2025,1,Baylor,Auburn,home,24.0,38.0,483,415,...,0.500000,0.0,311.0,-243.0,-2.0,-0.166667,-28.0,-396.0,1,0.441562
3485,401756847,2025,1,Kansas,Fresno State,home,31.0,7.0,383,216,...,0.466667,3.0,-3.0,170.0,9.0,-0.266667,22.0,376.0,1,0.985852
3486,401754523,2025,1,TCU,North Carolina,away,48.0,14.0,542,222,...,0.100000,2.0,112.0,208.0,19.0,0.483333,33.0,258.0,0,0.999592
3487,401757218,2025,1,Western Kentucky,Sam Houston,home,41.0,24.0,506,382,...,0.142857,-1.0,192.0,-68.0,12.0,0.436090,-34.0,352.0,1,0.883266
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4240,401777353,2025,15,Indiana,Ohio State,away,13.0,10.0,340,322,...,0.333333,0.0,-42.0,60.0,0.0,0.128205,29.0,-26.0,0,0.655756
4241,401777329,2025,15,Miami (OH),Western Michigan,away,13.0,23.0,272,397,...,0.266667,-1.0,88.0,-213.0,0.0,0.133333,13.0,-634.0,0,0.035404
4242,401777351,2025,15,Georgia,Alabama,away,28.0,7.0,297,209,...,0.214286,1.0,-56.0,144.0,5.0,0.160714,75.0,824.0,0,0.858471
4243,401777328,2025,15,Duke,Virginia,away,27.0,20.0,333,344,...,0.466667,1.0,-20.0,9.0,-7.0,-0.091667,32.0,528.0,0,0.498065


In [24]:
# make opponent win probability column with 1 - predicted_win_prob
predictions['opponent_win_prob'] = 1 - predictions['predicted_win_prob']

In [25]:
prediction_export = predictions[['game_id','team', 'opponent', 'predicted_win_prob', 'opponent_win_prob', 'home_away']].copy()

In [26]:
prediction_export.to_csv('predictions_2025.csv', index=False)